In [ ]:
# Databricks notebook source
# tutorial_name: 01-Day8-Databricks-Workflows-Jobs
# notebookName: 01-Day8-Databricks-Workflows-Jobs
# COMMAND ----------

# DBTITLE 1,Paths (same pattern as Days 5–7)
notepoint_rel = "hands-on/day-08/notebooks/01-Day8-Databricks-Workflows-Jobs.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16 (u01–u16)
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"
# Same naming style as day06-{STUDENT_ID}, day07-{STUDENT_ID} — no trailing slash on root
DAY8_ROOT = f"{BASE_PATH}day08-{STUDENT_ID}"
SMOKE_OUT = f"{DAY8_ROOT}/workflow_smoke/run_proof"
AUDIT_OUT = f"{DAY8_ROOT}/workflow_smoke/job_audit_runs"
# COMMAND ----------

# DBTITLE 1,Prerequisite check (aligned with Day 6 / Day 7)
print("DAY8_ROOT:", DAY8_ROOT)
print("SMOKE_OUT:", SMOKE_OUT)
print("AUDIT_OUT:", AUDIT_OUT)
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("✓ Day 5 tables found (P_BASIC)")
except Exception as e:
    print(f"✗ Day 5 tables not found: {e}")
    print("Complete Day 5 notebook 01 first.")
try:
    spark.read.format("csv").option("header", True).load(SOURCE_CSV).limit(1).collect()
    print("✓ SOURCE_CSV readable:", SOURCE_CSV)
except Exception as e:
    print(f"✗ SOURCE_CSV: {e}")
try:
    spark.read.format("json").load(SOURCE_JSON).limit(1).collect()
    print("✓ SOURCE_JSON readable:", SOURCE_JSON)
except Exception as e:
    print("(optional) SOURCE_JSON:", type(e).__name__)
# COMMAND ----------


# Day 8 — Item 21: Databricks Workflows (Jobs)

Syllabus item 21 — jobs concepts — **tasks**, **runs**, **clusters**; hands-on **create job pipeline**.

**Sequence with earlier days:** Uses the same **`BASE_PATH`**, **`DAY5`**, **`P_BASIC`**, **`SOURCE_CSV`** variables as **Day 5** / **Day 6** / **Day 7**. Writes only under **`day08-{STUDENT_ID}/`** so you do not overwrite `day05`, `day06`, or `day07` prefixes.

**Item 22:** start with **`02-Day8-Medallion-Guide-and-Student-Flow.ipynb`**, then task notebooks **`03`–`05`** (and optional **`06`**).


## Student flow — item 21 (follow in order)

1. In the **first code cell**, set **`STUDENT_ID`** to your assigned id, then run that cell through the prerequisite block. Fix any errors before continuing.
2. Run the rest of this notebook **from top to bottom** on one interactive cluster session. Later cells reuse variables such as **`JOB_CORRELATION`** and paths defined earlier.
3. When everything succeeds interactively, open **Workflows** → **Jobs** → **Create job**. Add **one** notebook task that points to **this** notebook file.
4. Choose compute (existing cluster for class, or a job cluster if your instructor prefers).
5. Click **Run now**, open the **Run** page, and confirm the task ends **Succeeded**.
6. Run the job again and verify **`AUDIT_OUT`** gains another row while **`SMOKE_OUT`** still reflects the latest overwrite.

Then open **`02-Day8-Medallion-Guide-and-Student-Flow.ipynb`** for item **22** (three separate task notebooks).


## Where jobs fit

Earlier days: you run PySpark in notebooks and write Delta on ABFS. **Day 8** adds **Jobs** so the same code can run on a **schedule**, from the **API**, or with **Run now**, with a clear **run history** in the UI.

The smoke Delta write in Lab A is ordinary Spark + Delta; attaching a **job** does not change the write logic — it only changes **who starts the run** and **how you observe** it.


## Objectives

- Name the main objects in **Workflows / Jobs**: job, task, run, trigger, compute policy.
- Run this notebook **interactively**, then bind it to a **single-task job** and compare behavior.
- Inspect **Delta** outputs and a simple **append-only audit** log for repeated job runs.
- Know when to use an **existing cluster** vs a **job cluster** (cost vs isolation).


## Path convention (matches Days 5–7)

| Variable | Example role |
|----------|----------------|
| `BASE_PATH` | `abfss://.../data/` — shared training container |
| `DAY5` | `BASE_PATH + "/day5"` — earlier labs |
| `P_BASIC` | Delta table from Day 5 |
| `DAY8_ROOT` | `BASE_PATH + "day08-" + STUDENT_ID` — **your** sandbox for today |

Outputs today: `.../day08-u25/workflow_smoke/...` (adjust `STUDENT_ID`).


## Why use a Job instead of only running a notebook?

| Interactive notebook | Job |
|----------------------|-----|
| You click **Run** | Scheduler or API triggers **Run now** |
| Stops when you disconnect | Intended for **production** and **SLAs** |
| Harder to audit centrally | **Runs** tab: history, duration, retries |
| Same code cells | Can pass **parameters**, use **task libraries** |

Jobs are how you **operationalize** the same PySpark you wrote on Days 2–7.


## Interactive notebook vs job run

- **Interactive:** you run cells; the cluster is usually already up; good for development.
- **Job:** the platform starts a run from **Run now** or a **schedule**; each attempt appears on the **Runs** tab with **logs** and **task status**.

Both use the same **`BASE_PATH`** and **`day08-{STUDENT_ID}`** paths when this notebook is the task.


## Glossary (UI-aligned)

- **Job** — Named container for tasks + optional schedules and notifications.
- **Task** — One step (e.g. **Notebook**, **Python file**, **ETL Pipeline**, **SQL**, **dbt**).
- **Run** — Single execution attempt; contains per-task status and logs.
- **Trigger** — Schedule (cron), **file arrival**, **continuous** (for some task types), or manual.
- **Compute** — Cluster or SQL warehouse attached to the task; can be **existing** or **job**-scoped.


## Common task types (high level)

1. **Notebook** — What you use in this course most often.
2. **Lakeflow ETL Pipelines** — Declarative pipelines (Day 7 `pipelines/`); created via **Create → ETL Pipeline** with **Lakeflow Pipelines Editor** and **Add existing assets** for the `.py` libraries (see `pipelines/DEPLOY.md`).
3. **SQL** — Warehouse query or file.
4. **Python / wheel** — Packaged tasks.

Multi-task graphs are item **22** (see notebook `02`).


## One job, one task, one run

A **job** holds configuration (name, tasks, triggers). A **run** is a single execution: the platform runs your **notebook task** on a cluster, then records **Succeeded** or **Failed** plus **logs**.

States you typically see: **Pending** → **Running** → **Succeeded** / **Failed** / **Cancelled** (with **Retrying** if retries are enabled).


In [ ]:
# Runtime context (useful when comparing interactive vs job runs)
import pyspark
print("Spark version:", spark.version)
print("Python:", pyspark.__version__)
try:
    cid = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "(not set)")
    print("ClusterId tag:", cid)
except Exception:
    pass


## Run lifecycle (what you see on the Runs page)

Typical states: **Pending** → **Running** → **Succeeded** or **Failed** / **Timeout** / **Cancelled**.  
If you enable **retries**, a failed task may transition to **Retrying** before final failure.

After you create a job, open **Run**, expand each **task**, and download **logs** when you need to debug a failure.


## Run states (concept)

Typical path: **Pending** (waiting for compute) → **Running** → **Succeeded** or **Failed**. If the job defines **retries**, a failed task may show **Retrying** before a final failure.


## Lab A — Write a deterministic “smoke” Delta table

This cell is the main **body** your **first job task** should execute. It **overwrites** `SMOKE_OUT` each time so reruns stay predictable.


In [ ]:
import uuid

JOB_CORRELATION = str(uuid.uuid4())[:12]
print("JOB_CORRELATION:", JOB_CORRELATION)

spark.sql(
    f"SELECT '{STUDENT_ID}' AS student_id, current_timestamp() AS ran_at, "
    f"'{JOB_CORRELATION}' AS correlation_id"
).write.format("delta").mode("overwrite").save(SMOKE_OUT)

spark.read.format("delta").load(SMOKE_OUT).show(truncate=False)


In [ ]:
# Exercise (concept check): build the SQL string the smoke cell will run — no write yet
demo_sql = (
    f"SELECT '{STUDENT_ID}' AS student_id, current_timestamp() AS ran_at, 'dry-run' AS correlation_id"
)
print(demo_sql)
spark.sql(demo_sql).show(truncate=False)


The cell above ran a **read-only** preview of the same shape as the smoke table. The next cell **materializes** it to Delta at `SMOKE_OUT`.


## Lab B — Delta history on the smoke table

Jobs should produce **observable** side effects. Use **`DESCRIBE HISTORY`** to see each job run as a new table version (if the write committed).


In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{SMOKE_OUT}`").select(
    "version", "operation", "operationMetrics"
).show(15, truncate=80)


## Delta history vs audit table

Each **overwrite** of **`SMOKE_OUT`** adds a new **Delta version** on that path. The **audit** path uses **append**, so one **row per job run** accumulates there — useful for a simple run registry.


## Lab C — Append-only audit log (multiple job runs)

Unlike `SMOKE_OUT` (overwrite), **`AUDIT_OUT`** uses **`append`** so repeated job runs accumulate rows.  
Use this pattern when you need a lightweight **run registry** without a separate database.


In [ ]:
import uuid
_corr = globals().get("JOB_CORRELATION") or str(uuid.uuid4())[:12]
audit_row = spark.sql(
    f"SELECT '{STUDENT_ID}' AS student_id, current_timestamp() AS logged_at, "
    f"'{_corr}' AS correlation_id, 'smoke_task' AS task_name"
)
audit_row.write.format("delta").mode("append").save(AUDIT_OUT)

print("Audit tail:")
spark.read.format("delta").load(AUDIT_OUT).orderBy("logged_at", ascending=False).show(10, truncate=False)


## Lab D — Table metadata (`DESCRIBE DETAIL`)

Useful in class: show **format**, **partitionColumns**, **numFiles** — same Delta vocabulary as **Day 5**.


In [ ]:
spark.sql(f"DESCRIBE DETAIL delta.`{SMOKE_OUT}`").show(truncate=False)


In [ ]:
# Exercise: compare row counts smoke vs audit (after Labs A–C)
try:
    s = spark.read.format('delta').load(SMOKE_OUT).count()
    a = spark.read.format('delta').load(AUDIT_OUT).count()
    print(f'smoke rows (latest overwrite): {s}')
    print(f'audit rows (cumulative appends): {a}')
except Exception as e:
    print('Run Labs A–C first:', e)


## Parameterizing jobs

For **item 22**, each medallion **stage** is a **separate notebook** (`03` / `04` / `05`), so you do **not** need a `pipeline_stage` widget — the job wires **three tasks** to **three files**.

For **this** notebook you can still add optional widgets (e.g. `dry_run`) using `dbutils.widgets.text(...)` and `dbutils.widgets.get(...)` if you want to experiment.


## Checklist — **single-task job** (item 21)

1. Run this notebook **interactively** top-to-bottom once; confirm no errors.
2. **Workflows** → **Jobs** → **Create job**.
3. Name: `day08_smoke_{STUDENT_ID}` (or your standard).
4. **Add task** → **Notebook** → pick **this notebook**.
5. **Compute:** existing cluster (class) or job cluster (production pattern).
6. **Run now** → wait for **Succeeded**.
7. Re-run the job → check **`AUDIT_OUT`** gained a new row; **`SMOKE_OUT`** still has one logical row (overwrite).
8. Optional: **Job settings** → **Notifications** on failure.
9. Optional: duplicate task (sequential) to simulate a two-step job before moving to **02**.


## Existing cluster vs job cluster

| | Existing / shared cluster | Job cluster |
|---|---------------------------|-------------|
| **Pros** | Fast start in class; shared cost | Isolated; autoscale; terminates when idle |
| **Cons** | Queuing; shared state | Cold start; need policy permissions |

For **production**, prefer **job clusters** created from a **policy** your admin defines.


## Permissions (typical)

- **CAN MANAGE RUN** on the job (or owner).
- **Attach** policy / cluster if using job compute.
- **READ/WRITE** on `abfss` paths (your app registration / credential chain from **`02-Mount-Azure-Data-Lake`** on other days).

If the job fails with **forbidden** on storage, fix **OAuth / service principal** the same way as Day 5 interactive runs.


## Troubleshooting

| Symptom | Things to check |
|---------|-----------------|
| Task **Pending** forever | Cluster not running; **max workers**; policy cap |
| **ImportError** in job | Cluster **libraries** differ from your notebook session |
| **Path not found** | `STUDENT_ID` typo; wrong `BASE_PATH`; storage ACL |
| **Delta** errors | Concurrent overwrite; run **VACUUM** only when taught (Day 6) |

**Extra reading:** `databricks/**/13-Building-n-Managing-Workflows-with-Databricks*`, `**/Job_Scheduling_and_Monitoring*`.


Open **`02-Day8-Medallion-Guide-and-Student-Flow.ipynb`** for item **22** (student flow and job wiring), then task notebooks **`03`–`05`** and optional **`06`** for post-run checks.


## Optional — schedule (concept)

Cron examples (UTC, illustrative):

- Daily 06:00: `0 0 6 * * ?`
- Every 6 hours: `0 0 */6 * * ?`

Configure in the job’s **Triggers / Schedule**; **pause** schedules in shared training workspaces when not teaching.


In [ ]:
# Optional cleanup (your prefix only) — uncomment in your workspace when needed
# dbutils.fs.rm(DAY8_ROOT, recurse=True)
